# 🎯 InternFinder — Interactive Demo Notebook

An agentic AI system that helps CS students find real internship postings.
This notebook walks through every component: configuration, resume parsing,
the agent's tool-calling loop, and the evaluation harness.

## Architecture Overview

```
  Student Prompt
       │
       ▼
  ┌─────────────┐
  │  Agent Loop  │ ← NVIDIA NIM (LLM)
  │  (agent.py)  │
  └──────┬───────┘
         │ calls tools?
         │
    ┌────┴────┐
    ▼         ▼
  ┌──────┐  ┌──────────┐
  │ Brave│  │  Other    │
  │Search│  │  Tools    │
  └───┬──┘  └──────────┘
      │
      ▼
  [INTERN_CARD] results
```

**Key files:**
- `config.py` — settings loaded from `.env`
- `clients.py` — shared async OpenAI client factory
- `prompts.py` — system prompts + StudentProfile
- `tools.py` — Brave Search tool definition
- `agent.py` — agentic while-loop with tool calling
- `resume_parser.py` — PDF/text → structured profile
- `main.py` — FastAPI server

## 1. Setup & Installation

If you haven't installed dependencies yet:
```bash
pip install -r requirements.txt
```

In [ ]:
import sys
import os
import json
import asyncio
from pathlib import Path

# Add the project root so we can import our modules
project_root = Path(__file__).parent
sys.path.insert(0, str(project_root))

# Suppress noisy loggers for cleaner notebook output
import logging
logging.getLogger('httpx').setLevel(logging.WARNING)
logging.getLogger('openai').setLevel(logging.WARNING)
logging.getLogger('asyncio').setLevel(logging.WARNING)

print(f"Project root: {project_root}")
print(f"Python: {sys.version}")
print(f"Modules: {', '.join(f for f in ['config', 'clients', 'prompts', 'tools', 'agent', 'resume_parser', 'main'] if Path(project_root / f'{f}.py').exists())}")

## 2. Configuration

The `Settings` dataclass loads from `.env`. Let's inspect what's configured.

In [ ]:
from config import settings

print("=== InternFinder Settings ===")
for attr in dir(settings):
    if not attr.startswith('_'):
        val = getattr(settings, attr)
        if val and attr == 'nvidia_api_key':
            # Mask the API key
            val = val[:8] + '...' + val[-4:] if len(val) > 12 else '***'
        elif val and attr == 'brave_api_key':
            val = val[:8] + '...' + val[-4:] if len(val) > 12 else '***'
        print(f"  {attr:20s} = {val}")

### 2a. Changing settings on the fly

You can tweak the model, token limits, or max tool rounds without editing `.env`:

In [ ]:
# Example: use a different model or adjust token limits
# settings.model = "meta/llama-3.1-405b-instruct"  # more capable
# settings.max_tokens = 2000  # longer responses
# settings.max_tool_rounds = 7  # allow more search rounds

print(f"Model: {settings.model}")
print(f"Max tokens: {settings.max_tokens}")
print(f"Max tool rounds: {settings.max_tool_rounds}")
print(f"API endpoint: {settings.base_url}")

## 3. StudentProfile — The Resume Data Model

`StudentProfile` is the structured representation of a student's resume.

In [ ]:
from prompts import StudentProfile

# Create a sample profile
demo_profile = StudentProfile(
    name="Jane Doe",
    degree="B.S. Computer Science",
    school="MIT",
    year="Junior",
    gpa="3.85",
    skills={
        "languages": ["Python", "Java", "C++"],
        "frameworks": ["FastAPI", "React", "TensorFlow"],
        "tools": ["Git", "Docker", "AWS", "Linux"],
        "other": ["Agile", "Technical Writing"],
    },
    active_skills=["Python", "FastAPI", "React"],  # user-selected active ones
    summary="Junior CS student with experience in web development and ML.",
)

print("=== Student Profile ===")
print(f"Name:    {demo_profile.name}")
print(f"Degree:  {demo_profile.degree}")
print(f"School:  {demo_profile.school}")
print(f"Year:    {demo_profile.year}")
print(f"GPA:     {demo_profile.gpa}")
print(f"Summary: {demo_profile.summary}")
print(f"\n=== Skills ===")
for cat, skills in demo_profile.skills.items():
    print(f"  {cat:12s}: {', '.join(skills)}")
print(f"\n=== Active Skills ===")
print(f"  {', '.join(demo_profile.active_skills)}")

## 4. System Prompts

The agent's behavior is defined entirely by prompts. Let's examine what the model sees.

In [ ]:
from prompts import build_system_prompt, SKILL_OUTPUT_FORMAT, SKILL_MATCH_SCORING, RESUME_PARSER_PROMPT

# Without a profile — general InternFinder behavior
base_prompt = build_system_prompt()
print("=== Base Prompt (no profile) ===")
print(base_prompt[:600] + "...\n")

# With a profile — personalized with student skills
personalized_prompt = build_system_prompt(demo_profile)
print("=== Personalized Prompt (with profile) ===")
print(personalized_prompt[:600] + "...\n")
print(f"Total prompt length: {len(personalized_prompt)} chars")
print(f"Contains [INTERN_CARD] format: {('[INTERN_CARD]' in personalized_prompt)}")
print(f"Contains student name: {demo_profile.name in personalized_prompt}")

### 4a. Guardrails built into the prompt

The system prompt enforces strict guardrails:

In [ ]:
guardrail_rules = [
    "ONLY discuss internships, jobs, careers, and student professional development",
    "NEVER fabricate companies, titles, or links",
    "If asked about anything unrelated, politely redirect",
    "If a location or pay requirement is unrealistic, say so",
]

print("=== Agent Guardrails ===\n")
for i, rule in enumerate(guardrail_rules, 1):
    print(f"{i}. {rule}")

print("\n=== Output Format (INTERN_CARD) ===")
print(SKILL_OUTPUT_FORMAT)

## 5. The Agent Loop

The agent's core is a while-loop that:
1. Sends the conversation to the LLM
2. If the LLM calls a tool (e.g., `brave_search`), executes it
3. Feeds tool results back to the LLM
4. Repeats until the LLM says `stop` or max rounds reached

Let's trace through what happens on a single query.

In [ ]:
from tools import TOOL_SCHEMAS

print("=== Available Tools ===")
for tool in TOOL_SCHEMAS:
    name = tool['function']['name']
    desc = tool['function']['description']
    params = tool['function']['parameters']
    print(f"\n{color_blue}{name}{color_reset}")
    print(f"  Description: {desc}")
    print(f"  Required params: {params.get('properties', {}).keys()}")

## 6. Chat Demo — Running the Agent

Now let's actually run the agent. It will search the live web for internship postings.

**Note:** This makes real API calls to NVIDIA NIM and Brave Search.

In [ ]:
from agent import run_agent

# Run the agent without a profile (general internship search)
print("Searching for software engineering internships...\n")

reply, updated_history = await run_agent(
    user_message="Find software engineering internships in Boston for juniors",
    conversation_history=[],
    profile=None,
)

print(f"Response length: {len(reply)} chars")
print(f"INTERN_CARDs found: {reply.count('[INTERN_CARD]')}")
print(f"\n--- First 800 chars ---\n{reply[:800]}..." if len(reply) > 800 else reply)

### 6a. Multi-turn conversation with a profile

With a student profile, the agent can score matches by skill relevance.

In [ ]:
# First turn — find internships
history = []
prompt1 = "Find remote data science internships for juniors with Python experience"

print(f"User: {prompt1}\n")
reply1, history = await run_agent(prompt1, history, demo_profile)
print(f"Agent: {reply1[:500]}..." if len(reply1) > 500 else f"Agent: {reply1}")
print(f"\nCards: {reply1.count('[INTERN_CARD]')}  |  History turns: {len(history)}")

In [ ]:
# Second turn — follow-up in the same conversation
prompt2 = "Can you find any cybersecurity internships instead?"

print(f"User: {prompt2}\n")
reply2, history = await run_agent(prompt2, history, demo_profile)
print(f"Agent: {reply2[:500]}..." if len(reply2) > 500 else f"Agent: {reply2}")
print(f"\nCards: {reply2.count('[INTERN_CARD]')}  |  History turns: {len(history)}")

### 6b. Guardrail test — off-topic prompt

The agent should politely redirect away from non-internship topics.

In [ ]:
off_topic = "What's the capital of France?"
reply = await run_agent(off_topic, [], None)
print(f"User: {off_topic}\nAgent: {reply[0]}")
print(f"\nHas INTERN_CARD: {'INTERN_CARD' in reply[0]}")

## 7. Resume Parsing

The resume parser extracts structured data from uploaded PDFs or text files using a two-stage pipeline:
1. **Text extraction** — deterministic (pypdf for PDFs, UTF-8 decode for text)
2. **LLM structuring** — the model outputs JSON matching the StudentProfile schema

In [ ]:
from resume_parser import parse_resume, extract_text_from_pdf, extract_text_from_plain

# Simulate a plain-text resume upload
sample_resume_text = """
John Smith
john.smith@mit.edu | (617) 555-0123 | linkedin.com/in/johnsmith

Education
Massachusetts Institute of Technology (MIT)
B.S. Computer Science and Artificial Intelligence
Expected Graduation: May 2027
GPA: 3.78 / 4.00
Relevant Coursework: Data Structures, Algorithms, Machine Learning,
  Operating Systems, Databases, Computer Networks

Experience
Software Engineering Intern — Google, Mountain View, CA (Summer 2024)
  • Built an internal dashboard using React and TypeScript
  • Optimized database queries reducing load time by 40%
  • Collaborated with a team of 5 engineers on Kubernetes deployment pipeline

Research Assistant — MIT CSAIL (Fall 2023 - Present)
  • Developed Python scripts for data preprocessing and analysis
  • Trained and evaluated BERT models using Hugging Face Transformers
  • Published a short paper on LLM fine-tuning techniques

Projects
  • Stock Price Predictor: LSTM model with 72% directional accuracy on S&P 500
  • Real-time Chat App: WebSocket-based messaging with Node.js and Redis

Skills
  Languages: Python, JavaScript, TypeScript, Java, SQL, HTML/CSS
  Frameworks: React, FastAPI, TensorFlow, Hugging Face, Node.js
  Tools: Git, Docker, Kubernetes, AWS (S3, EC2), Linux, PostgreSQL, MongoDB
  Other: Agile/Scrum, Technical Writing, Public Speaking
"""

print("=== Sample Resume Text ===")
print(sample_resume_text[:300] + "...\n")
print(f"Total characters: {len(sample_resume_text)}")

### 7a. Parse the resume through the LLM

In [ ]:
parsed_profile = await parse_resume(
    sample_resume_text.encode('utf-8'),
    content_type="text/plain",
)

print("=== Parsed Resume ===")
print(f"Name:    {parsed_profile.name}")
print(f"Degree:  {parsed_profile.degree}")
print(f"School:  {parsed_profile.school}")
print(f"Year:    {parsed_profile.year}")
print(f"GPA:     {parsed_profile.gpa}")
print(f"Summary: {parsed_profile.summary}")
print(f"\n=== Extracted Skills ===")
for cat, skills in parsed_profile.skills.items():
    print(f"  {cat:12s}: {', '.join(skills)}")

### 7b. What the LLM sees in the parser prompt

In [ ]:
print("=== Resume Parser System Prompt ===")
print(RESUME_PARSER_PROMPT[:700] + "...\n")
print(f"Total: {len(RESUME_PARSER_PROMPT)} chars")

## 8. End-to-End: Resume → Search Flow

The full pipeline: upload resume → get profile → search internships for that profile.

In [ ]:
# Step 1: Parse the resume
print("Step 1: Parsing resume...")
profile = await parse_resume(
    sample_resume_text.encode('utf-8'),
    content_type="text/plain",
)
print(f"  Extracted: {profile.name}, {profile.school}, {profile.year}")
print(f"  Skills: {', '.join(profile.skills.get('languages', []))}")

# Step 2: Set active skills (the user selects which skills to search with)
profile.active_skills = ["Python", "React", "FastAPI", "Docker"]
print(f"\nActive skills: {', '.join(profile.active_skills)}")

# Step 3: Search internships tailored to this profile
print("\nStep 3: Searching for personalized internships...")
reply, _ = await run_agent(
    "Find summer 2025 software engineering internships",
    [],
    profile,
)

print(f"\n=== Personalized Results ===")
print(f"Cards found: {reply.count('[INTERN_CARD]')}")
print(f"\n{reply[:1200]}..." if len(reply) > 1200 else reply)

## 9. Running the Evaluation Harness

The project includes an automated test suite. Let's run it.

In [ ]:
import json
from pathlib import Path

test_file = project_root / "evals" / "test_cases.json"
with open(test_file) as f:
    tests = json.load(f)['test_cases']

print(f"=== Eval Test Suite ({len(tests)} tests) ===\n")
for t in tests:
    badge = {"happy_path": "🟢", "edge_case": "🟡", "off_topic": "🔴"}.get(t['type'], '⚪')
    print(f"{badge} {t['id']}: {t['prompt'][:60]}{'...' if len(t['prompt']) > 60 else ''}")
    if t.get('note'):
        print(f"   → {t['note']}")

In [ ]:
from agent import run_agent

# Run a few key tests (skip the ones that require actual API calls if desired)
# Run all tests
GREEN = "\033[92m"
RED = "\033[91m"
YELLOW = "\033[93m"
CYAN = "\033[96m"
RESET = "\033[0m"
BOLD = "\033[1m"

results = []
for test in tests:
    print(f"\n{CYAN}[{test['id']}]{RESET} {BOLD}{test['prompt']}{RESET}")
    if test.get('note'):
        print(f"  {YELLOW}expect: {test['note']}{RESET}")
    try:
        resp, _ = await run_agent(test['prompt'], [], None)
    except Exception as e:
        print(f"{RED}✗ EXCEPTION: {e}{RESET}")
        results.append(False)
        continue

    failed = []
    resp_lower = resp.lower()
    for phrase in test.get('expect_contains', []):
        if phrase.lower() not in resp_lower:
            failed.append(f"missing '{phrase}'")
    for phrase in test.get('expect_not_contains', []):
        if phrase.lower() in resp_lower:
            failed.append(f"forbidden '{phrase}' found")
    card_count = resp.count('[INTERN_CARD]')
    if card_count < test.get('min_cards', 0):
        failed.append(f"cards: {card_count} < {test.get('min_cards', 0)}")

    if not failed:
        print(f"{GREEN}✓ PASSED{RESET}")
        results.append(True)
    else:
        print(f"{RED}✗ FAILED: {', '.join(failed)}{RESET}")
        results.append(False)

passed = sum(results)
print(f"\n{BOLD}Results: {GREEN}{passed} passed{RESET} / {len(results)} total{RESET}")

## 10. The Agent Tool-Calling Loop — Step by Step

Let's peek inside the agent's loop to see how it handles tool calls.

In [ ]:
from openai import AsyncOpenAI
from clients import nim_client
from prompts import build_system_prompt
from tools import TOOL_SCHEMAS

# Manually run one turn to see the raw model response
client = nim_client()
system_prompt = build_system_prompt()

print("=== Raw Model Response (first turn) ===\n")
print("Sending: 'Find Python internships in New York'\n")

resp = await client.chat.completions.create(
    model=settings.model,
    max_tokens=settings.max_tokens,
    temperature=0.2,
    tools=TOOL_SCHEMAS,
    messages=[
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": "Find Python internships in New York"},
    ],
)

msg = resp.choices[0].message
finish_reason = resp.choices[0].finish_reason

print(f"Finish reason: {finish_reason}")
print(f"Has tool calls: {bool(msg.tool_calls)}")

if msg.tool_calls:
    for tc in msg.tool_calls:
        print(f"\nTool call #{tc.id}:")
        print(f"  Function: {tc.function.name}")
        print(f"  Arguments: {tc.function.arguments}")
else:
    content = msg.content or ""
    print(f"\nResponse text (first 200 chars):\n{content[:200]}" if content else "(no content)")

## 11. Quick Start — Running the Full Server

If you want to run the complete FastAPI server from this notebook:

In [ ]:
# Uncomment to start the server in the background:
# import subprocess
# server_process = subprocess.Popen(
#     [sys.executable, "-m", "uvicorn", "main:app", "--port", "8000"],
#     stdout=subprocess.PIPE,
#     stderr=subprocess.PIPE,
# )
# print(f"Server started on PID {server_process.pid}")
# print("API docs: http://localhost:8000/docs")

# To stop the server later:
# server_process.terminate()
# server_process.wait()

## 12. Testing via HTTP (when server is running)

If the server is running, you can test it with these requests.

In [ ]:
import httpx

BASE_URL = "http://localhost:8000"

# Health check
try:
    r = httpx.get(f"{BASE_URL}/")
    print(f"Health: {r.json()}")
except Exception as e:
    print(f"Server not running — start with: uvicorn main:app --reload --port 8000")
    print(f"Error: {e}")

In [ ]:
# Chat via HTTP
try:
    r = httpx.post(
        f"{BASE_URL}/chat",
        json={
            "message": "Find data science internships in San Francisco",
            "active_skills": ["Python", "Machine Learning"],
        },
        timeout=30,
    )
    data = r.json()
    print(f"Session ID: {data['session_id']}")
    print(f"\nReply:\n{data['reply']}")
except Exception as e:
    print(f"Server not running. Error: {e}")

## 13. Production Checklist

Before deploying to production, consider:

In [ ]:
production_checklist = [
    ("Session storage", "Replace in-memory dict with Redis or PostgreSQL"),
    ("CORS", "Lock down allow_origins to your frontend domain"),
    ("API key management", "Use secrets manager (AWS Secrets Manager, HashiCorp Vault)"),
    ("Rate limiting", "Add middleware to prevent API abuse"),
    ("Authentication", "Add JWT/auth middleware before agent access"),
    ("Logging", "Use structured JSON logs with a central log aggregator"),
    ("Monitoring", "Add metrics endpoint (Prometheus) for latency, error rate, token usage"),
    ("Deployment", "Run behind Gunicorn with multiple workers for concurrency"),
    ("File uploads", "Add antivirus scan for uploaded resumes"),
    ("Cost tracking", "Log token usage per request for billing"),
]

print("=== Production Checklist ===\n")
for i, (item, detail) in enumerate(production_checklist, 1):
    print(f"{i}. {item}")
    print(f"   → {detail}")